In [1]:
# ============================================================
# CELL 1 — CÀI ĐẶT
# ============================================================
!pip install -q mediapipe opencv-python-headless numpy matplotlib

!wget -O face_landmarker.task -q \
    https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

!pip install -q fastapi uvicorn pyngrok nest-asyncio firebase-admin python-dotenv httpx python-multipart

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared


print("✓ FastAPI + cloudflared OK")
print("✓ Cài đặt hoàn tất")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.0 MB/s eta 0:00:00
✓ FastAPI + cloudflared OK
✓ Cài đặt hoàn tất


In [2]:
# ============================================================
# CELL 2 — DATA CONTRACTS
# ============================================================
from dataclasses import dataclass
from typing import Optional, List
import numpy as np


@dataclass
class FrameData:
    timestamp_ms:          int
    blendshapes:           dict
    landmarks:             list
    transformation_matrix: Optional[np.ndarray] = None


@dataclass
class AffectResult:
    duchenne_ratio:         float   # [0,1] genuine_events / total_smile_events
    flat_affect_score:      float   # [0,1] 1 - variance blendshapes
    smiling_duration_ratio: float   # [0,1]
    happy_face_ratio:       float   # [0,1]


@dataclass
class GazeResult:
    gaze_instability:     float   # [0,1] std(yaw) / YAW_REF
    gaze_break_rate:      float   # [0,1] break events/phút / BREAK_REF
    gaze_avoidance_score: float = 0.0
    center_gaze_ratio:    float = 0.0   # deprecated


@dataclass
class HeadPoseResult:
    head_down_ratio:        float   # [0,1]
    head_movement_variance: float


@dataclass
class BlinkResult:
    blink_rate:         float   # blinks/giây
    avg_blink_duration: float   # giây — frame_skip-invariant


@dataclass
class EmotionResult:
    # --- 6 chỉ số lưu Firestore ---
    duchenne_ratio:        float
    flat_affect_score:     float
    gaze_instability:      float
    head_down_ratio:       float
    blink_duration_avg:    float   # seconds
    behavioral_risk_score: float
    # --- Extended ---
    smiling_duration_ratio: float = 0.0
    happy_face_ratio:       float = 0.0
    gaze_break_rate:        float = 0.0
    gaze_avoidance_score:   float = 0.0
    head_movement_variance: float = 0.0
    blink_rate:             float = 0.0
    frames_processed:       int   = 0
    confidence:             float = 0.0

print("✓ Data contracts OK")

✓ Data contracts OK


In [3]:
# ============================================================
# CELL 3 — MODULE A: Face Landmarker
# extract_landmarks_and_blendshapes(video_path) → List[FrameData]
# ============================================================
import cv2
import mediapipe as mp


def extract_landmarks_and_blendshapes(
    video_path: str,
    frame_skip: int = 1,
    model_path: str = "face_landmarker.task",
) -> List[FrameData]:
    BaseOptions        = mp.tasks.BaseOptions
    FaceLandmarker     = mp.tasks.vision.FaceLandmarker
    FaceLandmarkerOpts = mp.tasks.vision.FaceLandmarkerOptions
    RunningMode        = mp.tasks.vision.RunningMode

    options = FaceLandmarkerOpts(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=RunningMode.VIDEO,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
        num_faces=1,
    )
    frames_data: List[FrameData] = []

    with FaceLandmarker.create_from_options(options) as landmarker:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Không mở được video: {video_path}")
        fps       = cap.get(cv2.CAP_PROP_FPS) or 30.0
        frame_idx = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % frame_skip == 0:
                ts_ms  = int(frame_idx * 1000 / fps)
                rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result = landmarker.detect_for_video(mp_img, ts_ms)

                if result.face_landmarks:
                    bs: dict = {}
                    if result.face_blendshapes:
                        for b in result.face_blendshapes[0]:
                            bs[b.category_name] = b.score
                    tm: Optional[np.ndarray] = None
                    if result.facial_transformation_matrixes:
                        tm = np.array(
                            result.facial_transformation_matrixes[0].data
                        ).reshape(4, 4)
                    frames_data.append(FrameData(
                        timestamp_ms=ts_ms,
                        blendshapes=bs,
                        landmarks=result.face_landmarks[0],
                        transformation_matrix=tm,
                    ))
            frame_idx += 1
        cap.release()

    return frames_data

print("✓ Module A OK")

✓ Module A OK


In [4]:
# ============================================================
# CELL 4 — MODULE B: Affect Analyzer
# compute_affect_indicators(frames) → AffectResult
# ============================================================

def compute_affect_indicators(frames: List[FrameData]) -> AffectResult:
    """
    Duchenne Ratio — event-based:
      Phát hiện SMILE EVENTS (chuỗi frames ms > SMILE_THRESHOLD).
      genuine nếu: length >= 8 frames AND peak_cheek >= 0.35
                   AND apex_cheek >= peak_smile * 0.60
      duchenne_ratio = genuine_events / total_events  → [0,1]

    Constants calibrated từ 8 video test thực tế:
      CHEEK_THRESHOLD = 0.35  (tăng từ 0.20 vì baseline cheekSquint
                               73–100% frames ở mọi video)
      MIN_SMILE_EVENTS = 2    (guard: < 2 events → ratio không đáng tin)

    Flat Affect:
      top-15 blendshapes biến động nhất × sensitivity 5.0
    """
    SMILE_THRESHOLD   = 0.15
    CHEEK_THRESHOLD   = 0.35
    DUCHENNE_COUPLING = 0.60
    MIN_SMILE_FRAMES  = 8
    MIN_SMILE_EVENTS  = 2
    FLAT_SENSITIVITY  = 5.0
    FLAT_TOP_N        = 15

    if not frames:
        return AffectResult(0.0, 1.0, 0.0, 0.0)

    ms_series, cs_series, all_bs_matrix = [], [], []
    for f in frames:
        bs = f.blendshapes
        ms = (bs.get("mouthSmileLeft", 0.0) + bs.get("mouthSmileRight", 0.0)) / 2.0
        cs = max(
            (bs.get("cheekSquintLeft", 0.0) + bs.get("cheekSquintRight", 0.0)) / 2.0,
            (bs.get("eyeSquintLeft",   0.0) + bs.get("eyeSquintRight",   0.0)) / 2.0,
        )
        ms_series.append(ms)
        cs_series.append(cs)
        if bs:
            all_bs_matrix.append(list(bs.values()))

    # Smile event detection
    smile_events: List[dict] = []
    in_smile = False
    event_ms, event_cs, event_start = [], [], 0

    for i, (ms, cs) in enumerate(zip(ms_series, cs_series)):
        if ms > SMILE_THRESHOLD:
            if not in_smile:
                in_smile, event_start, event_ms, event_cs = True, i, [], []
            event_ms.append(ms)
            event_cs.append(cs)
        else:
            if in_smile:
                smile_events.append({
                    "length":  i - event_start,
                    "ms_list": event_ms,
                    "cs_list": event_cs,
                })
                in_smile = False
    if in_smile:
        smile_events.append({
            "length":  len(ms_series) - event_start,
            "ms_list": event_ms,
            "cs_list": event_cs,
        })

    total_events         = len(smile_events)
    genuine_events       = 0
    total_smile_frames   = 0
    genuine_smile_frames = 0

    for ev in smile_events:
        length  = ev["length"]
        peak_ms = max(ev["ms_list"])
        peak_cs = max(ev["cs_list"])
        apex_cs = ev["cs_list"][ev["ms_list"].index(peak_ms)]
        total_smile_frames += length
        if (length  >= MIN_SMILE_FRAMES
                and peak_cs >= CHEEK_THRESHOLD
                and apex_cs >= peak_ms * DUCHENNE_COUPLING):
            genuine_events       += 1
            genuine_smile_frames += length

    # Guard: quá ít events → ratio không đáng tin
    duchenne_ratio = (genuine_events / total_events
                      if total_events >= MIN_SMILE_EVENTS else 0.0)

    n = len(frames)

    if len(all_bs_matrix) > 1:
        bs_mat   = np.array(all_bs_matrix)
        top_stds = sorted(np.std(bs_mat, axis=0), reverse=True)[:FLAT_TOP_N]
        flat     = float(np.clip(1.0 - np.mean(top_stds) * FLAT_SENSITIVITY, 0.0, 1.0))
    else:
        flat = 1.0

    return AffectResult(
        duchenne_ratio=         round(float(np.clip(duchenne_ratio, 0.0, 1.0)), 4),
        flat_affect_score=      round(flat, 4),
        smiling_duration_ratio= round(total_smile_frames   / n, 4),
        happy_face_ratio=       round(genuine_smile_frames / n, 4),
    )

print("✓ Module B OK")

✓ Module B OK


In [5]:
# ============================================================
# CELL 5 — MODULE C: Gaze Analyzer
# compute_gaze_indicators(frames) → GazeResult
# ============================================================

def compute_gaze_indicators(frames: List[FrameData]) -> GazeResult:
    """
    Gaze analysis cho selfie scenario dùng head yaw angle.

    Lý do không dùng iris-ratio:
      std ~0.047, biên độ [0.47, 0.52] — mọi threshold cho ~96% center.

    Lý do không dùng center_gaze_ratio:
      Selfie: camera = điểm nhìn tự nhiên → luôn ≈ 0.95+.

    Hai metrics thay thế:
      1. gaze_instability = std(yaw) / YAW_REF
      2. gaze_break_rate:
           - Đếm EVENTS (transitions), không đếm frames
           - relative_yaw = yaw - person_mean  (loại bỏ baseline tự nhiên)
           - personal_break_thr = max(2*std, 10°)  (2-sigma cá nhân hóa)
    """
    YAW_REF   = 15.0
    BREAK_REF = 20.0   # events/phút

    if not frames:
        return GazeResult(gaze_instability=0.0, gaze_break_rate=0.0)

    yaws: List[float] = []
    for f in frames:
        if f.transformation_matrix is None:
            continue
        R = f.transformation_matrix[:3, :3]
        yaws.append(float(np.degrees(
            np.arctan2(-R[2, 0], np.sqrt(R[2, 1]**2 + R[2, 2]**2))
        )))

    if not yaws:
        return GazeResult(gaze_instability=0.0, gaze_break_rate=0.0)

    yaw_arr  = np.array(yaws)
    yaw_std  = float(yaw_arr.std())
    yaw_mean = float(yaw_arr.mean())

    instability        = float(np.clip(yaw_std / YAW_REF, 0.0, 1.0))
    personal_break_thr = max(2.0 * yaw_std, 10.0)
    relative_yaws      = yaw_arr - yaw_mean

    break_events = 0
    in_break     = False
    for rv in relative_yaws:
        if abs(rv) > personal_break_thr:
            if not in_break:
                break_events += 1
                in_break      = True
        else:
            in_break = False

    dur_min    = (frames[-1].timestamp_ms - frames[0].timestamp_ms) / 60_000.0 if len(frames) > 1 else 1.0
    break_rate = float(np.clip((break_events / dur_min) / BREAK_REF, 0.0, 1.0)) if dur_min > 0 else 0.0
    avoidance  = float(np.clip((instability + break_rate) / 2.0, 0.0, 1.0))

    return GazeResult(
        gaze_instability=     round(instability,  4),
        gaze_break_rate=      round(break_rate,   4),
        gaze_avoidance_score= round(avoidance,    4),
    )

print("✓ Module C OK")


✓ Module C OK


In [6]:
# ============================================================
# CELL 6 — MODULE D: Head Pose Analyzer
# compute_head_pose_indicators(frames) → HeadPoseResult
# ============================================================

def compute_head_pose_indicators(frames: List[FrameData]) -> HeadPoseResult:
    """
    pitch = arctan2(R[2,1], R[2,2])
      + = cúi xuống  |  - = ngẩng lên
    HEAD_DOWN_THR = 15°
    """
    HEAD_DOWN_THR = 15.0

    pitches: List[float] = []
    for f in frames:
        if f.transformation_matrix is None:
            continue
        R = f.transformation_matrix[:3, :3]
        pitches.append(float(np.degrees(np.arctan2(R[2, 1], R[2, 2]))))

    if not pitches:
        return HeadPoseResult(head_down_ratio=0.0, head_movement_variance=0.0)

    diffs = [abs(pitches[i] - pitches[i - 1]) for i in range(1, len(pitches))]

    return HeadPoseResult(
        head_down_ratio=        round(sum(1 for p in pitches if p > HEAD_DOWN_THR) / len(pitches), 4),
        head_movement_variance= round(float(np.std(diffs)) if diffs else 0.0, 4),
    )

print("✓ Module D OK")

✓ Module D OK


In [7]:
# ============================================================
# CELL 7 — MODULE E: Blink Analyzer
# compute_blink_indicators(frames) → BlinkResult
# ============================================================

def compute_blink_indicators(frames: List[FrameData]) -> BlinkResult:
    """
    Adaptive threshold (fix từ data thực 8 videos):
      threshold = clip(mean + 2.0*std, 0.45, 0.85)

    Guard: max(eyeBlink) < 0.40 → signal quá yếu → return 0

    avg_blink_duration tính bằng giây (frame_skip-invariant):
      eff_fps = 1000 / median(timestamp_intervals)

    MAX_BLINK_DURATION_S = 0.50s:
      Blink sinh lý tối đa ~400ms (Stern et al. 1984)
      Event > 0.5s → loại khỏi avg (không phải blink thật)
    """
    MAX_BLINK_DURATION_S = 0.50
    MIN_RELIABLE_MAX     = 0.40

    if not frames:
        return BlinkResult(blink_rate=0.0, avg_blink_duration=0.0)

    if len(frames) > 1:
        intervals     = [frames[i].timestamp_ms - frames[i - 1].timestamp_ms
                         for i in range(1, len(frames))]
        median_ms     = float(np.median(intervals))
        effective_fps = 1000.0 / median_ms if median_ms > 0 else 30.0
    else:
        effective_fps = 30.0

    scores = np.array([
        (f.blendshapes.get("eyeBlinkLeft", 0.0) + f.blendshapes.get("eyeBlinkRight", 0.0)) / 2.0
        for f in frames
    ])

    if scores.max() < MIN_RELIABLE_MAX:
        return BlinkResult(blink_rate=0.0, avg_blink_duration=0.0)

    adaptive_thr = float(np.clip(scores.mean() + 2.0 * scores.std(), 0.45, 0.85))

    in_blink    = False
    blink_count = 0
    valid_durs: List[float] = []
    cur_len     = 0

    for s in scores:
        if s > adaptive_thr:
            if not in_blink:
                in_blink, cur_len = True, 0
            cur_len += 1
        else:
            if in_blink:
                dur          = cur_len / effective_fps
                blink_count += 1
                if dur <= MAX_BLINK_DURATION_S:
                    valid_durs.append(dur)
                in_blink = False

    if in_blink and cur_len > 0:
        dur          = cur_len / effective_fps
        blink_count += 1
        if dur <= MAX_BLINK_DURATION_S:
            valid_durs.append(dur)

    total_s = (frames[-1].timestamp_ms - frames[0].timestamp_ms) / 1000.0 if len(frames) > 1 else 1.0

    return BlinkResult(
        blink_rate=         round(blink_count / total_s if total_s > 0 else 0.0, 4),
        avg_blink_duration= round(float(np.mean(valid_durs)) if valid_durs else 0.0, 4),
    )

print("✓ Module E OK")

✓ Module E OK


In [8]:
# ============================================================
# CELL 8 — MODULE F: Risk Scorer
# compute_behavioral_risk(...) → float
# ============================================================

def compute_behavioral_risk(
    affect:           AffectResult,
    gaze:             GazeResult,
    head:             HeadPoseResult,
    blink:            BlinkResult,
    population_stats: dict | None = None,
) -> float:
    """
    Weights proportional với feature importance Figure 5 (Kayış 2025),
    điều chỉnh cho selfie context:

      Paper → Selfie:
        center_gaze 0.30 → gaze_proxy  0.15  (signal yếu hơn)
        blink_dur   0.22 → blink_dur   0.22
        happy_face  0.22 → happy_face  0.25
        flat_affect 0.16 → flat_affect 0.22
        head_down   0.10 → head_down   0.16

    Cấp 1 (pilot, population_stats=None):
      Normalization bằng literature reference.
      BLINK_REF_S = 0.40s (Stern et al. 1984)
      HC_HAPPY    = 0.35  (ước lượng từ Kayış 2025)

    Cấp 2 (có data pilot):
      Truyền population_stats dict → z-score + sigmoid.
      Keys: blink_mean, blink_std, gaze_mean, gaze_std,
            happy_mean, happy_std, flat_mean, flat_std,
            head_mean, head_std
    """
    WEIGHTS = {
        "s_gaze":  0.15,
        "s_blink": 0.22,
        "s_happy": 0.25,
        "s_flat":  0.22,
        "s_head":  0.16,
    }
    BLINK_REF_S = 0.40
    HC_HAPPY    = 0.35

    def clip01(v: float) -> float:
        return float(np.clip(v, 0.0, 1.0))

    def z_sigmoid(val, mean, std, invert=False):
        if std < 1e-6:
            return 0.5
        z = (val - mean) / std
        if invert:
            z = -z
        return float(1.0 / (1.0 + np.exp(-z)))

    if population_stats is not None:
        s_gaze  = z_sigmoid(gaze.gaze_avoidance_score,
                            population_stats.get("gaze_mean",  0.30),
                            population_stats.get("gaze_std",   0.15))
        s_blink = z_sigmoid(blink.avg_blink_duration,
                            population_stats.get("blink_mean", 0.25),
                            population_stats.get("blink_std",  0.10))
        s_happy = z_sigmoid(affect.happy_face_ratio,
                            population_stats.get("happy_mean", 0.20),
                            population_stats.get("happy_std",  0.12),
                            invert=True)
        s_flat  = z_sigmoid(affect.flat_affect_score,
                            population_stats.get("flat_mean",  0.50),
                            population_stats.get("flat_std",   0.15))
        s_head  = z_sigmoid(head.head_down_ratio,
                            population_stats.get("head_mean",  0.10),
                            population_stats.get("head_std",   0.10))
    else:
        s_gaze  = clip01(gaze.gaze_avoidance_score)
        s_blink = clip01(blink.avg_blink_duration / BLINK_REF_S)
        s_happy = clip01((HC_HAPPY - affect.happy_face_ratio) / HC_HAPPY)
        s_flat  = clip01(affect.flat_affect_score)
        s_head  = clip01(head.head_down_ratio)

    signals = {
        "s_gaze": s_gaze, "s_blink": s_blink,
        "s_happy": s_happy, "s_flat": s_flat, "s_head": s_head,
    }
    risk = sum(WEIGHTS[k] * signals[k] for k in WEIGHTS)
    return round(float(np.clip(risk, 0.0, 1.0)), 4)

print("✓ Module F OK")

✓ Module F OK


In [9]:
# ============================================================
# CELL 9 — ORCHESTRATOR
# analyze_selfie_video(video_path) → EmotionResult
# ============================================================

def analyze_selfie_video(
    video_path:  str,
    frame_skip:  int        = 1,
    model_path:  str        = "face_landmarker.task",
    verbose:     bool       = True,
    pop_stats:   dict|None  = None,
) -> EmotionResult:
    if verbose:
        print(f"[A] Extracting landmarks: {video_path}")
    frames = extract_landmarks_and_blendshapes(video_path, frame_skip, model_path)

    if not frames:
        raise ValueError("Không phát hiện khuôn mặt trong video.")
    if verbose:
        dur = (frames[-1].timestamp_ms - frames[0].timestamp_ms) / 1000
        print(f"    → {len(frames)} frames / {dur:.1f}s")

    if verbose: print("[B] Computing affect indicators...")
    affect = compute_affect_indicators(frames)

    if verbose: print("[C] Computing gaze indicators...")
    gaze   = compute_gaze_indicators(frames)

    if verbose: print("[D] Computing head pose indicators...")
    head   = compute_head_pose_indicators(frames)

    if verbose: print("[E] Computing blink indicators...")
    blink  = compute_blink_indicators(frames)

    if verbose: print("[F] Computing behavioral risk score...")
    risk   = compute_behavioral_risk(affect, gaze, head, blink, pop_stats)

    confidence = float(np.clip(len(frames) / 150.0, 0.1, 1.0))

    return EmotionResult(
        duchenne_ratio=         affect.duchenne_ratio,
        flat_affect_score=      affect.flat_affect_score,
        gaze_instability=       gaze.gaze_instability,
        head_down_ratio=        head.head_down_ratio,
        blink_duration_avg=     blink.avg_blink_duration,
        behavioral_risk_score=  risk,
        smiling_duration_ratio= affect.smiling_duration_ratio,
        happy_face_ratio=       affect.happy_face_ratio,
        gaze_break_rate=        gaze.gaze_break_rate,
        gaze_avoidance_score=   gaze.gaze_avoidance_score,
        head_movement_variance= head.head_movement_variance,
        blink_rate=             blink.blink_rate,
        frames_processed=       len(frames),
        confidence=             round(confidence, 2),
    )

print("✓ Orchestrator OK")

✓ Orchestrator OK


In [10]:
# ============================================================
# CELL 10 — API ADAPTER + FIRESTORE PAYLOAD
# ============================================================
import json


def to_firestore_payload(result: EmotionResult) -> dict:
    """
    6 chỉ số lưu Firestore.
    Tất cả [0,1] trừ blink_duration_avg_s (seconds).
    """
    return {
        "duchenne_ratio":        result.duchenne_ratio,
        "flat_affect_score":     result.flat_affect_score,
        "gaze_instability":      result.gaze_instability,
        "head_down_ratio":       result.head_down_ratio,
        "blink_duration_avg_s":  result.blink_duration_avg,
        "behavioral_risk_score": result.behavioral_risk_score,
    }


# FastAPI skeleton (dùng khi deploy)
"""
from fastapi import FastAPI, UploadFile
import shutil, uuid, os

app = FastAPI()

@app.post("/analyze-emotion")
async def analyze_emotion(file: UploadFile):
    tmp = f"/tmp/{uuid.uuid4()}.mp4"
    with open(tmp, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        result  = analyze_selfie_video(tmp, verbose=False)
        payload = to_firestore_payload(result)
        return {"status": "ok", "data": payload, "confidence": result.confidence}
    finally:
        os.remove(tmp)
"""

print("✓ API adapter OK")

✓ API adapter OK


In [11]:
# ============================================================
# CELL 11 — DISPLAY
# display_emotion_report(result)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def display_emotion_report(result: EmotionResult) -> None:
    BLINK_REF_S = 0.40

    plot_fields = {
        "duchenne_ratio":        result.duchenne_ratio,
        "flat_affect_score":     result.flat_affect_score,
        "gaze_instability":      result.gaze_instability,
        "head_down_ratio":       result.head_down_ratio,
        "blink_duration_avg":    min(result.blink_duration_avg / BLINK_REF_S, 1.0),
        "behavioral_risk_score": result.behavioral_risk_score,
    }
    labels = {
        "duchenne_ratio":        "Duchenne Ratio\n(nụ cười thật ↑=tốt)",
        "flat_affect_score":     "Flat Affect\n(biểu cảm phẳng ↑=nguy cơ)",
        "gaze_instability":      "Gaze Instability\n(đầu dao động ↑=nguy cơ)",
        "head_down_ratio":       "Head Down\n(cúi đầu ↑=nguy cơ)",
        "blink_duration_avg":    "Blink Duration\n(norm, ↑=nguy cơ)",
        "behavioral_risk_score": "Behavioral Risk\n(tổng hợp ↑=nguy cơ)",
    }
    risk_keys = {"flat_affect_score", "head_down_ratio",
                 "blink_duration_avg", "behavioral_risk_score", "gaze_instability"}
    good_keys = {"duchenne_ratio"}

    def bar_color(key, val):
        if key in risk_keys:
            return "#d62728" if val > 0.60 else "#ff7f0e" if val > 0.35 else "#2ca02c"
        if key in good_keys:
            return "#2ca02c" if val > 0.55 else "#ff7f0e" if val > 0.30 else "#d62728"
        return "#7f7f7f"

    fig, ax = plt.subplots(figsize=(9, 5))
    keys   = list(plot_fields.keys())[::-1]
    vals   = [plot_fields[k] for k in keys]
    colors = [bar_color(k, v) for k, v in zip(keys, vals)]
    lbls   = [labels[k] for k in keys]

    bars = ax.barh(lbls, vals, color=colors, edgecolor="white", height=0.55)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{v:.3f}", va="center", fontsize=9)

    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Score (0–1)")
    ax.set_title(
        f"MIND Emotion Analysis  |  Risk: {result.behavioral_risk_score:.2f}  "
        f"|  Confidence: {result.confidence*100:.0f}%  "
        f"|  Frames: {result.frames_processed}",
        fontweight="bold", pad=10,
    )
    ax.legend(handles=[
        mpatches.Patch(color="#2ca02c", label="Tốt"),
        mpatches.Patch(color="#ff7f0e", label="Chú ý"),
        mpatches.Patch(color="#d62728", label="Nguy cơ"),
    ], fontsize=8, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    print("\n=== Firestore Payload ===")
    print(json.dumps(to_firestore_payload(result), indent=2))
    print(f"\nConfidence        : {result.confidence:.0%}  |  Frames: {result.frames_processed}")
    print(f"Gaze break rate   : {result.gaze_break_rate:.4f}")
    print(f"Gaze avoidance    : {result.gaze_avoidance_score:.4f}")
    print(f"Smiling duration  : {result.smiling_duration_ratio:.4f}")
    print(f"Happy face ratio  : {result.happy_face_ratio:.4f}")
    print(f"Head movement var : {result.head_movement_variance:.4f}")
    print(f"Blink rate (b/s)  : {result.blink_rate:.4f}")

print("✓ Display OK")

✓ Display OK


In [12]:
# ============================================================
# CELL 12 — BATCH RUNNER
# Đặt video vào thư mục INPUT_FOLDER rồi chạy cell này
# ============================================================
import os, glob, cv2


def show_video_preview(video_path: str) -> None:
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if ret:
        plt.figure(figsize=(5, 3))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Preview: {os.path.basename(video_path)}", fontsize=9)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(f"[!] Không đọc được frame: {video_path}")


# ── Cấu hình ─────────────────────────────────────────────────
INPUT_FOLDER = "videos"   # tạo thư mục này và upload video vào
FRAME_SKIP   = 1          # 1 = toàn bộ | 2 = video dài > 30s
VIDEO_EXTS   = ("*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm")

# Population stats — bật khi có data pilot
# POP_STATS = {
#     "blink_mean": 0.20, "blink_std": 0.08,
#     "gaze_mean":  0.30, "gaze_std":  0.15,
#     "happy_mean": 0.20, "happy_std": 0.12,
#     "flat_mean":  0.40, "flat_std":  0.15,
#     "head_mean":  0.10, "head_std":  0.08,
# }
POP_STATS = None

# ── Chạy ─────────────────────────────────────────────────────
if not os.path.exists(INPUT_FOLDER):
    print(f"[ERROR] Thư mục '{INPUT_FOLDER}' không tồn tại.")
else:
    video_files = []
    for ext in VIDEO_EXTS:
        video_files.extend(glob.glob(os.path.join(INPUT_FOLDER, ext)))

    if not video_files:
        print(f"Không tìm thấy video nào trong '{INPUT_FOLDER}'.")
    else:
        print(f"Tìm thấy {len(video_files)} video.\n")
        summary = []

        for vp in sorted(video_files):
            print("=" * 70)
            print(f"{'  ' + os.path.basename(vp) + '  ':█^70}")
            try:
                show_video_preview(vp)
                result = analyze_selfie_video(
                    vp,
                    frame_skip=FRAME_SKIP,
                    verbose=True,
                    pop_stats=POP_STATS,
                )
                display_emotion_report(result)
                summary.append((os.path.basename(vp), result))

            except Exception as e:
                import traceback
                print(f"[ERROR] {vp}: {e}")
                traceback.print_exc()

        if summary:
            print("\n" + "=" * 70)
            print("SUMMARY")
            print(f"{'Video':<32} {'Risk':>6} {'Duch':>6} {'Flat':>6} {'Blink_s':>8} {'Gaze':>6}")
            print("-" * 70)
            for name, r in summary:
                print(
                    f"{name[:30]:<32} {r.behavioral_risk_score:>6.3f} "
                    f"{r.duchenne_ratio:>6.3f} {r.flat_affect_score:>6.3f} "
                    f"{r.blink_duration_avg:>8.4f} {r.gaze_instability:>6.4f}"
                )
            print("=" * 70)
            print("HOÀN THÀNH.")

[ERROR] Thư mục 'videos' không tồn tại.


In [13]:
import os
from dotenv import load_dotenv

load_dotenv("/content/.env")

FIREBASE_CREDENTIALS_PATH = os.environ["FIREBASE_CREDENTIALS_PATH"]

print("✓ Config OK")

✓ Config OK


In [14]:
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(FIREBASE_CREDENTIALS_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✓ Firebase OK")

✓ Firebase OK


In [15]:
import uuid
from datetime import datetime, timezone
from fastapi import FastAPI, UploadFile, File, HTTPException, Depends, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from firebase_admin import auth
import shutil, tempfile

app = FastAPI(title="MIND Backend API", version="2.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

_bearer = HTTPBearer()
ALLOWED_MIME   = {"video/mp4", "video/quicktime", "video/x-msvideo", "video/webm"}
MAX_SIZE_BYTES = 200 * 1024 * 1024


def get_current_user(
    credentials: HTTPAuthorizationCredentials = Depends(_bearer),
) -> dict:
    try:
        return auth.verify_id_token(credentials.credentials)
    except auth.ExpiredIdTokenError:
        raise HTTPException(status_code=401, detail="Token expired")
    except auth.InvalidIdTokenError:
        raise HTTPException(status_code=401, detail="Invalid token")
    except Exception as e:
        raise HTTPException(status_code=401, detail=str(e))


def classify_risk(score: float) -> dict:
    if score < 0.35:
        return {"level": "safe",     "label": "Tốt",    "color": "#2ca02c"}
    elif score <= 0.60:
        return {"level": "warning",  "label": "Chú ý",  "color": "#ff7f0e"}
    else:
        return {"level": "high_risk","label": "Nguy cơ","color": "#d62728"}


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/auth/me")
def get_me(payload: dict = Depends(get_current_user)):
    uid = payload["uid"]
    ref = db.collection("users").document(uid)
    if not ref.get().exists:
        ref.set({
            "user_id":    uid,
            "email":      payload.get("email"),
            "name":       payload.get("name"),
            "picture":    payload.get("picture"),
            "created_at": datetime.now(timezone.utc).isoformat(),
        })
    return {
        "user_id": uid,
        "email":   payload.get("email"),
        "name":    payload.get("name"),
        "picture": payload.get("picture"),
    }


@app.post("/videos/upload")
async def upload_video(
    file: UploadFile = File(...),
    payload: dict = Depends(get_current_user),
):
    if file.content_type not in ALLOWED_MIME:
        raise HTTPException(status_code=415, detail=f"Unsupported file type: {file.content_type}")

    content = await file.read()
    if len(content) > MAX_SIZE_BYTES:
        raise HTTPException(status_code=413, detail="Video exceeds 200 MB limit")

    # Lưu tạm vào /tmp rồi gọi thẳng hàm analyzer — cùng notebook nên không cần HTTP
    suffix = os.path.splitext(file.filename or "video.mp4")[1] or ".mp4"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        tmp_path = tmp.name

    try:
        result  = analyze_selfie_video(tmp_path, verbose=False)
        payload_data = to_firestore_payload(result)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        os.remove(tmp_path)

    video_id = str(uuid.uuid4())
    uid      = payload["uid"]
    risk     = classify_risk(result.behavioral_risk_score)

    doc = {
        "video_id":    video_id,
        "user_id":     uid,
        "result":      payload_data,
        "risk":        risk,
        "confidence":  result.confidence,
        "frames":      result.frames_processed,
        "analyzed_at": datetime.now(timezone.utc).isoformat(),
    }

    db.collection("analysis_results").document(video_id).set(doc)
    db.collection("users").document(uid).collection("analyses").document(video_id).set(doc)

    return {
        "video_id":    video_id,
        "result":      doc["result"],
        "risk":        doc["risk"],
        "confidence":  doc["confidence"],
        "frames":      doc["frames"],
        "analyzed_at": doc["analyzed_at"],
    }


@app.get("/history/me")
def get_my_history(
    limit: int = Query(default=20, ge=1, le=100),
    payload: dict = Depends(get_current_user),
):
    uid  = payload["uid"]
    docs = (
        db.collection("users").document(uid).collection("analyses")
        .order_by("analyzed_at", direction="DESCENDING")
        .limit(limit)
        .stream()
    )
    return {"user_id": uid, "records": [d.to_dict() for d in docs]}


@app.get("/history/me/{video_id}")
def get_single_result(
    video_id: str,
    payload: dict = Depends(get_current_user),
):
    uid = payload["uid"]
    doc = db.collection("users").document(uid).collection("analyses").document(video_id).get()
    if not doc.exists:
        raise HTTPException(status_code=404, detail="Result not found")
    return doc.to_dict()


print("✓ App defined OK")

✓ App defined OK


In [16]:
import uvicorn
import threading
import subprocess
import re
import time
import nest_asyncio

PORT = 8000

nest_asyncio.apply()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

url = None
for _ in range(30):
    line = proc.stderr.readline().decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print("=" * 60)
    print(f"  BACKEND URL = {url}")
    print("=" * 60)
    print("  → Dùng URL này trong Postman và app Java")
else:
    print("[ERROR] Không bắt được URL.")
    print(proc.stderr.read().decode("utf-8", errors="ignore")[:1000])

  BACKEND URL = https://hold-broadway-authorized-fortune.trycloudflare.com
  → Dùng URL này trong Postman và app Java
